# Temporal Visualization of Gender Role Sentiment Analysis

Analyzing sentiment patterns over time for 女主内 (female role) and 男主外 (male role) discourse on Weibo from 2016-2023.

## Section 1: Load and Inspect Result Data

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load both datasets
nvzhunei_df = pd.read_csv('results/nvzhunei_qwen_analysis_results.csv')
nanzhuwai_df = pd.read_csv('results/nanzhuwai_qwen_analysis_results.csv')

# Add a label to distinguish them
nvzhunei_df['dataset'] = '女主内'
nanzhuwai_df['dataset'] = '男主外'

# Combine datasets
df_combined = pd.concat([nvzhunei_df, nanzhuwai_df], ignore_index=True)

print(f"女主内 (Female Role) - {len(nvzhunei_df)} posts")
print(f"男主外 (Male Role) - {len(nanzhuwai_df)} posts")
print(f"Total combined: {len(df_combined)} posts\n")

print("Column names:")
print(df_combined.columns.tolist())
print("\nData types:")
print(df_combined.dtypes)
print("\nFirst few rows:")
print(df_combined[['time', 'year', 'month', 'qwen_sentiment', 'qwen_bucket', 'dataset']].head(10))

女主内 (Female Role) - 1683 posts
男主外 (Male Role) - 1544 posts
Total combined: 3227 posts

Column names:
['post_id', 'time', 'text', 'user_id', 'repost_text', 'source_file', 'cleaned_text', 'cleaned_repost', 'full_text', 'content_hash', 'text_length', 'year', 'month', 'is_relevant', 'processed_at', 'qwen_sentiment', 'qwen_bucket', 'qwen_confidence', 'qwen_reasoning', 'qwen_error', 'qwen_processed_at', 'dataset']

Data types:
post_id                int64
time                  object
text                  object
user_id                int64
repost_text           object
source_file           object
cleaned_text          object
cleaned_repost        object
full_text             object
content_hash           int64
text_length            int64
year                 float64
month                float64
is_relevant             bool
processed_at          object
qwen_sentiment       float64
qwen_bucket           object
qwen_confidence      float64
qwen_reasoning       float64
qwen_error            o

## Section 2: Parse Timestamp Fields and Sort Chronologically

In [3]:
# Parse time column
df_combined['time'] = pd.to_datetime(df_combined['time'], errors='coerce')

# Create year-month column for easier aggregation
df_combined['year_month'] = df_combined['time'].dt.to_period('M')
df_combined['date'] = df_combined['time'].dt.date

# Fill missing year/month from time column
df_combined['year'] = df_combined['year'].fillna(df_combined['time'].dt.year).astype('Int64')
df_combined['month'] = df_combined['month'].fillna(df_combined['time'].dt.month).astype('Int64')

# Sort by time
df_combined = df_combined.sort_values('time').reset_index(drop=True)

# Remove rows with NaN sentiment (errors)
df_clean = df_combined.dropna(subset=['qwen_sentiment']).copy()
df_clean['qwen_sentiment'] = df_clean['qwen_sentiment'].astype(int)

print(f"Posts with valid sentiment: {len(df_clean)} / {len(df_combined)}")
print(f"Error rate: {100 * (1 - len(df_clean) / len(df_combined)):.1f}%")
print(f"\nDate range: {df_clean['time'].min()} to {df_clean['time'].max()}")
print(f"\nSentiment distribution:")
print(df_clean['qwen_sentiment'].value_counts().sort_index())

Posts with valid sentiment: 2950 / 3227
Error rate: 8.6%

Date range: 2016-01-01 10:20:00 to 2025-04-01 23:34:54

Sentiment distribution:
qwen_sentiment
-2    1311
-1    1105
 0     423
 1      65
 2      46
Name: count, dtype: int64


## Section 3: Aggregate Results by Time Interval

In [4]:
# Monthly aggregation by dataset
monthly_stats = df_clean.groupby(['year_month', 'dataset']).agg({
    'qwen_sentiment': ['count', 'mean', 'std'],
    'post_id': 'count'
}).reset_index()

monthly_stats.columns = ['year_month', 'dataset', 'count', 'avg_sentiment', 'sentiment_std', 'total_posts']
monthly_stats['year_month'] = monthly_stats['year_month'].dt.to_timestamp()
monthly_stats = monthly_stats.sort_values('year_month')

# Sentiment breakdown by month and dataset
sentiment_by_month = df_clean.groupby(['year_month', 'dataset', 'qwen_sentiment']).size().reset_index(name='count')
sentiment_by_month['year_month'] = sentiment_by_month['year_month'].dt.to_timestamp()

# Bucket distribution by month and dataset
bucket_by_month = df_clean.dropna(subset=['qwen_bucket']).groupby(['year_month', 'dataset', 'qwen_bucket']).size().reset_index(name='count')
bucket_by_month['year_month'] = bucket_by_month['year_month'].dt.to_timestamp()

print("Monthly statistics summary:")
print(monthly_stats.head(10))
print(f"\nTotal months: {len(monthly_stats['year_month'].unique())}")

Monthly statistics summary:
  year_month dataset  count  avg_sentiment  sentiment_std  total_posts
0 2016-01-01     女主内      1       0.000000            NaN            1
1 2016-01-01     男主外      2      -1.000000       1.414214            2
2 2016-02-01     女主内      1       0.000000            NaN            1
3 2016-03-01     男主外      2       0.000000       0.000000            2
4 2016-04-01     男主外      2       0.000000       1.414214            2
5 2016-05-01     女主内      1      -2.000000            NaN            1
6 2016-05-01     男主外      3      -0.666667       1.154701            3
7 2016-06-01     女主内      6      -1.333333       0.816497            6
8 2016-06-01     男主外      3      -0.666667       0.577350            3
9 2016-07-01     女主内      2       0.000000       0.000000            2

Total months: 112


## Section 4: Plot Temporal Trends - Sentiment Over Time

In [13]:
# Overall sentiment distribution by dataset
sentiment_by_dataset = df_clean.groupby(['dataset', 'qwen_sentiment']).size().reset_index(name='count')

sentiment_map = {-2: 'Strongly Negative', -1: 'Negative', 0: 'Neutral', 1: 'Positive', 2: 'Strongly Positive'}
color_map = {-2: '#d62728', -1: '#ff7f0e', 0: '#7f7f7f', 1: '#2ca02c', 2: '#1f77b4'}

fig1 = go.Figure()

for dataset in ['女主内', '男主外']:
    data = sentiment_by_dataset[sentiment_by_dataset['dataset'] == dataset]
    data = data.sort_values('qwen_sentiment')
    fig1.add_trace(go.Bar(
        x=[sentiment_map[int(s)] for s in data['qwen_sentiment']],
        y=data['count'],
        name=dataset,
        hovertemplate='<b>%{fullData.name}</b><br>%{x}<br>Count: %{y}<extra></extra>'
    ))

fig1.update_layout(
    title='Total Sentiment Distribution by Dataset',
    xaxis_title='Sentiment',
    yaxis_title='Number of Posts',
    barmode='group',
    height=500,
    template='plotly_white',
    hovermode='x unified'
)

fig1.show()

In [7]:
# Post volume over time
fig2 = go.Figure()

for dataset in ['女主内', '男主外']:
    data = monthly_stats[monthly_stats['dataset'] == dataset]
    fig2.add_trace(go.Bar(
        x=data['year_month'],
        y=data['count'],
        name=dataset,
        hovertemplate='<b>%{fullData.name}</b><br>Month: %{x|%Y-%m}<br>Posts: %{y}<extra></extra>'
    ))

fig2.update_layout(
    title='Monthly Post Volume by Dataset',
    xaxis_title='Time',
    yaxis_title='Number of Posts',
    barmode='group',
    height=500,
    template='plotly_white',
    hovermode='x unified'
)

fig2.show()

In [8]:
# Stacked sentiment distribution over time
fig3 = make_subplots(rows=2, cols=1, subplot_titles=('女主内 (Female Role)', '男主外 (Male Role)'), 
                      specs=[[{'secondary_y': False}], [{'secondary_y': False}]],
                      vertical_spacing=0.12)

sentiment_map = {-2: 'Strongly Negative', -1: 'Negative', 0: 'Neutral', 1: 'Positive', 2: 'Strongly Positive'}
color_map = {-2: '#d62728', -1: '#ff7f0e', 0: '#7f7f7f', 1: '#2ca02c', 2: '#1f77b4'}

for row, dataset in enumerate(['女主内', '男主外'], 1):
    data = sentiment_by_month[sentiment_by_month['dataset'] == dataset]
    
    for sentiment in sorted(data['qwen_sentiment'].unique()):
        sentiment_data = data[data['qwen_sentiment'] == sentiment]
        fig3.add_trace(
            go.Bar(
                x=sentiment_data['year_month'],
                y=sentiment_data['count'],
                name=sentiment_map.get(sentiment, f'Sentiment {sentiment}'),
                marker_color=color_map.get(sentiment, '#555'),
                hovertemplate='<b>%{fullData.name}</b><br>Month: %{x|%Y-%m}<br>Count: %{y}<extra></extra>',
                showlegend=(row == 1)
            ),
            row=row, col=1
        )

fig3.update_layout(
    title_text='Sentiment Distribution Over Time by Dataset',
    height=700,
    barmode='stack',
    template='plotly_white',
    hovermode='x unified'
)

fig3.update_xaxes(title_text='Time', row=2, col=1)
fig3.update_yaxes(title_text='Number of Posts', row=1, col=1)
fig3.update_yaxes(title_text='Number of Posts', row=2, col=1)

fig3.show()

## Section 5: Compare Metrics Across Time Periods

In [17]:
# Yearly statistics by dataset
yearly_stats = df_clean.groupby(['year', 'dataset']).agg({
    'qwen_sentiment': ['count', 'mean', lambda x: (x > 0).sum(), lambda x: (x < 0).sum(), lambda x: (x == 0).sum()],
}).reset_index()

yearly_stats.columns = ['year', 'dataset', 'total_posts', 'avg_sentiment', 'positive_count', 'negative_count', 'neutral_count']
yearly_stats['positive_pct'] = 100 * yearly_stats['positive_count'] / yearly_stats['total_posts']
yearly_stats['negative_pct'] = 100 * yearly_stats['negative_count'] / yearly_stats['total_posts']
yearly_stats['neutral_pct'] = 100 * yearly_stats['neutral_count'] / yearly_stats['total_posts']

print("Yearly Statistics by Dataset:")
print(yearly_stats.to_string())

# Graph 1: Yearly post volume comparison
fig4a = go.Figure()

for dataset in ['女主内', '男主外']:
    data = yearly_stats[yearly_stats['dataset'] == dataset]
    fig4a.add_trace(
        go.Bar(x=data['year'], y=data['total_posts'], name=dataset,
               hovertemplate='<b>%{fullData.name}</b><br>Year: %{x}<br>Posts: %{y}<extra></extra>')
    )

fig4a.update_layout(
    title='Yearly Post Volume by Dataset',
    xaxis_title='Year',
    yaxis_title='Number of Posts',
    barmode='group',
    height=500,
    template='plotly_white',
    hovermode='x unified'
)

fig4a.show()

# Graph 2: Yearly sentiment comparison
fig4b = go.Figure()

for dataset in ['女主内', '男主外']:
    data = yearly_stats[yearly_stats['dataset'] == dataset]
    fig4b.add_trace(
        go.Scatter(x=data['year'], y=data['avg_sentiment'], name=dataset,
                   mode='lines+markers', line=dict(width=3), marker=dict(size=10),
                   hovertemplate='<b>%{fullData.name}</b><br>Year: %{x}<br>Avg Sentiment: %{y:.2f}<extra></extra>')
    )

fig4b.update_layout(
    title='Yearly Average Sentiment by Dataset',
    xaxis_title='Year',
    yaxis_title='Average Sentiment',
    height=500,
    template='plotly_white',
    hovermode='x unified'
)

fig4b.show()

Yearly Statistics by Dataset:
    year dataset  total_posts  avg_sentiment  positive_count  negative_count  neutral_count  positive_pct  negative_pct  neutral_pct
0   2016     女主内           53      -1.132075               1              41             11      1.886792     77.358491    20.754717
1   2016     男主外           67      -0.925373               4              46             17      5.970149     68.656716    25.373134
2   2017     女主内           76      -1.210526               4              63              9      5.263158     82.894737    11.842105
3   2017     男主外           83      -1.240964               3              70             10      3.614458     84.337349    12.048193
4   2018     女主内          144      -1.187500               7             119             18      4.861111     82.638889    12.500000
5   2018     男主外          148      -1.175676               5             123             20      3.378378     83.108108    13.513514
6   2019     女主内          159      -1.1

In [14]:
# Bucket/Topic distribution by dataset
bucket_total = df_clean.dropna(subset=['qwen_bucket']).groupby(['dataset', 'qwen_bucket']).size().reset_index(name='count')
bucket_total = bucket_total.sort_values('count', ascending=True)

fig5 = go.Figure()

for dataset in ['女主内', '男主外']:
    data = bucket_total[bucket_total['dataset'] == dataset]
    fig5.add_trace(go.Bar(
        y=data['qwen_bucket'],
        x=data['count'],
        name=dataset,
        orientation='h',
        hovertemplate='<b>%{fullData.name}</b><br>%{y}<br>Count: %{x}<extra></extra>'
    ))

fig5.update_layout(
    title='Topic/Bucket Distribution by Dataset',
    xaxis_title='Number of Posts',
    yaxis_title='Topic/Bucket',
    barmode='group',
    height=500,
    template='plotly_white',
    hovermode='x unified'
)

fig5.show()

## Section 6: Build an Interactive Time-Based Visualization

In [15]:
# Simplified dashboard with key metrics
fig_dash = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Monthly Post Count', 'Sentiment Distribution', 'Buckets by Dataset', 'Posts by Year'),
    specs=[[{'secondary_y': False}, {'secondary_y': False}],
           [{'secondary_y': False}, {'secondary_y': False}]]
)

# Plot 1: Monthly volume
for dataset in ['女主内', '男主外']:
    data = monthly_stats[monthly_stats['dataset'] == dataset]
    fig_dash.add_trace(
        go.Bar(x=data['year_month'], y=data['count'], name=dataset,
               hovertemplate='<b>%{fullData.name}</b><br>%{x|%Y-%m}: %{y} posts<extra></extra>',
               showlegend=True),
        row=1, col=1
    )

# Plot 2: Sentiment counts (from previous cell data)
for dataset in ['女主内', '男主外']:
    data = sentiment_by_dataset[sentiment_by_dataset['dataset'] == dataset].sort_values('qwen_sentiment')
    fig_dash.add_trace(
        go.Bar(x=[sentiment_map[int(s)] for s in data['qwen_sentiment']], 
               y=data['count'], name=f'{dataset}',
               hovertemplate='<b>%{fullData.name}</b><br>%{x}: %{y}<extra></extra>',
               showlegend=False),
        row=1, col=2
    )

# Plot 3: Buckets
for dataset in ['女主内', '男主外']:
    data = bucket_total[bucket_total['dataset'] == dataset]
    fig_dash.add_trace(
        go.Bar(y=data['qwen_bucket'], x=data['count'], name=dataset,
               orientation='h', showlegend=False,
               hovertemplate='<b>%{fullData.name}</b><br>%{y}: %{x}<extra></extra>'),
        row=2, col=1
    )

# Plot 4: Yearly totals
yearly_totals = df_clean.groupby(['year', 'dataset']).size().reset_index(name='count')
for dataset in ['女主内', '男主外']:
    data = yearly_totals[yearly_totals['dataset'] == dataset]
    fig_dash.add_trace(
        go.Bar(x=data['year'], y=data['count'], name=dataset,
               hovertemplate='<b>%{fullData.name}</b><br>%{x}: %{y} posts<extra></extra>',
               showlegend=False),
        row=2, col=2
    )

fig_dash.update_xaxes(title_text='Time', row=1, col=1)
fig_dash.update_xaxes(title_text='Sentiment', row=1, col=2)
fig_dash.update_xaxes(title_text='Posts', row=2, col=1)
fig_dash.update_xaxes(title_text='Year', row=2, col=2)

fig_dash.update_yaxes(title_text='Count', row=1, col=1)
fig_dash.update_yaxes(title_text='Count', row=1, col=2)
fig_dash.update_yaxes(title_text='Topic', row=2, col=1)
fig_dash.update_yaxes(title_text='Count', row=2, col=2)

fig_dash.update_layout(
    title_text='Analysis Overview: Sentiment & Topic Distribution',
    height=900,
    showlegend=True,
    hovermode='closest',
    template='plotly_white',
    barmode='group'
)

fig_dash.show()

In [12]:
# Summary statistics and insights
print("=" * 70)
print("ANALYSIS SUMMARY: Gender Role Discourse on Weibo (2016-2023)")
print("=" * 70)

for dataset in ['女主内', '男主外']:
    data = df_clean[df_clean['dataset'] == dataset]
    print(f"\n{dataset} Dataset:")
    print(f"  Total Posts Analyzed: {len(data)}")
    print(f"  Date Range: {data['time'].min().date()} to {data['time'].max().date()}")
    print(f"  Average Sentiment: {data['qwen_sentiment'].mean():.2f}")
    print(f"  Sentiment Distribution:")
    for sent in sorted(data['qwen_sentiment'].unique()):
        count = (data['qwen_sentiment'] == sent).sum()
        pct = 100 * count / len(data)
        print(f"    {sentiment_map.get(sent, f'Level {sent}')}: {count:4d} ({pct:5.1f}%)")
    
    print(f"  Average Confidence: {data['qwen_confidence'].mean():.1f}%")
    print(f"  Topic Distribution:")
    for bucket in data['qwen_bucket'].value_counts().head(3).index:
        count = (data['qwen_bucket'] == bucket).sum()
        pct = 100 * count / len(data)
        print(f"    {bucket}: {count:4d} ({pct:5.1f}%)")

print("\n" + "=" * 70)
print("KEY FINDINGS:")
print("=" * 70)

# Trend analysis
nvzhunei_trend = yearly_stats[yearly_stats['dataset'] == '女主内'].sort_values('year')
nanzhuwai_trend = yearly_stats[yearly_stats['dataset'] == '男主外'].sort_values('year')

if len(nvzhunei_trend) > 1:
    nvzhunei_change = nvzhunei_trend['avg_sentiment'].iloc[-1] - nvzhunei_trend['avg_sentiment'].iloc[0]
    print(f"\n女主内 (Female Role) Trend: {'↑ More Positive' if nvzhunei_change > 0 else '↓ More Negative'} over time ({nvzhunei_change:+.2f})")

if len(nanzhuwai_trend) > 1:
    nanzhuwai_change = nanzhuwai_trend['avg_sentiment'].iloc[-1] - nanzhuwai_trend['avg_sentiment'].iloc[0]
    print(f"男主外 (Male Role) Trend: {'↑ More Positive' if nanzhuwai_change > 0 else '↓ More Negative'} over time ({nanzhuwai_change:+.2f})")

print("\nPeak Activity Months:")
peak_month = monthly_stats.loc[monthly_stats['count'].idxmax()]
print(f"  {peak_month['dataset']}: {peak_month['year_month'].strftime('%Y-%m')} ({int(peak_month['count'])} posts)")

print("\nHighest Sentiment Months:")
highest_sentiment = monthly_stats.loc[monthly_stats['avg_sentiment'].idxmax()]
print(f"  {highest_sentiment['dataset']}: {highest_sentiment['year_month'].strftime('%Y-%m')} ({highest_sentiment['avg_sentiment']:.2f})")

print("\n" + "=" * 70)

ANALYSIS SUMMARY: Gender Role Discourse on Weibo (2016-2023)

女主内 Dataset:
  Total Posts Analyzed: 1532
  Date Range: 2016-01-01 to 2025-04-01
  Average Sentiment: -1.19
  Sentiment Distribution:
    Strongly Negative:  652 ( 42.6%)
    Negative:  601 ( 39.2%)
    Neutral:  226 ( 14.8%)
    Positive:   27 (  1.8%)
    Strongly Positive:   26 (  1.7%)
  Average Confidence: 88.9%
  Topic Distribution:
    个人立场/体验:  815 ( 53.2%)
    社会批判:  540 ( 35.2%)
    文化/历史/社会分析:   81 (  5.3%)

男主外 Dataset:
  Total Posts Analyzed: 1418
  Date Range: 2016-01-01 to 2025-04-01
  Average Sentiment: -1.23
  Sentiment Distribution:
    Strongly Negative:  659 ( 46.5%)
    Negative:  504 ( 35.5%)
    Neutral:  197 ( 13.9%)
    Positive:   38 (  2.7%)
    Strongly Positive:   20 (  1.4%)
  Average Confidence: 89.2%
  Topic Distribution:
    个人立场/体验:  710 ( 50.1%)
    社会批判:  533 ( 37.6%)
    文化/历史/社会分析:   84 (  5.9%)

KEY FINDINGS:

女主内 (Female Role) Trend: ↓ More Negative over time (-0.09)
男主外 (Male Role) Tr